# RAG Module · Day 20 — **RAG Architecture Patterns**

**Phase 1 · Module 3: Prompt Engineering & RAG Architecture** · ~2.5 hrs

Yesterday you engineered the *prompt*; today you engineer what the prompt **sees**. A RAG system's answer quality is capped by its *retrieval* quality, and retrieval quality is decided by two choices most teams get wrong: **how you chunk** the documents, and **how you rank** them.

Today:
1. **Naive → Advanced → Modular RAG** — the three generations, and what each adds.
2. **Chunking strategies** — fixed, recursive, semantic — and *measuring* which retrieves better.
3. **Retrieval metrics** — `recall@k` and **MRR**, the numbers that tell you a chunking change helped.
4. **Advanced techniques** — query expansion that rescues a failing query.
5. **Embedding-model choice** — Titan vs `text-embedding-3-small`.

> **Runnable keyless:** this morning's `cosine_similarity`, a deterministic bag-of-words embedder standing in for a real embedding model, and LangChain's real `RecursiveCharacterTextSplitter`. No API key.

## 1. Three generations of RAG

| | **Naive RAG** | **Advanced RAG** | **Modular RAG** |
|---|---|---|---|
| Pipeline | chunk → embed → retrieve top-k → stuff into prompt | + pre-retrieval (query rewrite/expansion) and post-retrieval (rerank, compress) | swappable modules wired as a graph |
| Retrieval | single vector search | hybrid (vector + keyword), reranking | routing, multi-hop, tool calls |
| Fails when | bad chunks, ambiguous queries, no reranking | rigid pipeline, one-size-fits-all | complexity/latency if over-built |
| Barclays fit | a demo | most production RAG | complex multi-source agents (Day 18's graph is here) |

The progression is **additive** — Advanced RAG is Naive plus fixes for its failure modes; Modular RAG makes those fixes *pluggable*. Everything below builds Naive, measures where it fails, then adds one Advanced fix.

## 2. Chunking — the decision that caps retrieval quality

Embeddings are computed per **chunk**, so chunk boundaries decide what can be retrieved as a unit. Three strategies:

- **Fixed-size** — slice every N characters. Simple, but cuts through the middle of sentences, splitting a fact from its context.
- **Recursive** — split on a hierarchy of separators (paragraph → sentence → word), keeping semantic units intact. LangChain's default; what Bedrock Knowledge Bases use.
- **Semantic** — pack whole sentences up to a size budget, never splitting one.

The document below is a policy page; watch where each strategy cuts.

In [1]:
import re, numpy as np
from langchain_text_splitters import RecursiveCharacterTextSplitter

DOC = ("Residential mortgages require a minimum deposit of five percent. "
       "The maximum loan-to-value ratio for a residential mortgage is 95 percent. "
       "Buy-to-let mortgages require a minimum deposit of 25 percent. "
       "Applicants must be at least 18 years old to hold a personal loan. "
       "The debt-to-income ratio must not exceed 45 percent for approval. "
       "A credit score below 580 is referred to manual underwriting. "
       "Overdraft facilities are capped at 2000 pounds for standard accounts. "
       "Business loans require a minimum trading history of 24 months.")

def fixed_chunks(text, size=90):
    return [text[i:i+size] for i in range(0, len(text), size)]

def recursive_chunks(text, size=90):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=size, chunk_overlap=0, separators=[". ", "? ", "\n", " ", ""])
    return splitter.split_text(text)

def semantic_chunks(text, size=120):
    sents, out, cur = re.split(r"(?<=[.?]) ", text), [], ""
    for s in sents:
        if len(cur) + len(s) <= size:
            cur = (cur + " " + s).strip()
        else:
            out.append(cur); cur = s
    if cur: out.append(cur)
    return out

print("--- FIXED (cuts mid-sentence) ---")
for c in fixed_chunks(DOC)[:3]:
    print(" |", c)
print("\n--- SEMANTIC (whole sentences) ---")
for c in semantic_chunks(DOC)[:3]:
    print(" |", c)

--- FIXED (cuts mid-sentence) ---
 | Residential mortgages require a minimum deposit of five percent. The maximum loan-to-value
 |  ratio for a residential mortgage is 95 percent. Buy-to-let mortgages require a minimum de
 | posit of 25 percent. Applicants must be at least 18 years old to hold a personal loan. The

--- SEMANTIC (whole sentences) ---
 | Residential mortgages require a minimum deposit of five percent.
 | The maximum loan-to-value ratio for a residential mortgage is 95 percent.
 | Buy-to-let mortgages require a minimum deposit of 25 percent.


☝ Fixed-size splits *`...maximum loan-to-value`* | *`ratio for a residential mortgage is 95 percent`* — the fact `95 percent` is now in a different chunk from `loan-to-value`, so a query for the LTV limit may never retrieve them together. Semantic/recursive keep each rule whole. That difference is measurable — next.

## 3. Measure it — `recall@k` and **MRR**

Two standard retrieval metrics, computed against a **golden set** (queries with a known-correct answer chunk):

- **recall@k** — did a relevant chunk appear in the top *k*? (Did we retrieve it *at all*?)
- **MRR** (Mean Reciprocal Rank) — 1/rank of the first relevant chunk, averaged. Rewards putting the right chunk **first** (rank 1 → 1.0, rank 3 → 0.33).

We embed each chunk (bag-of-words stand-in for a real embedder), rank by this morning's cosine, and score the three chunkers.

In [2]:
def cosine_similarity(a, b):
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom else 0.0

def make_embedder(chunks):
    vocab = sorted({w for c in chunks for w in re.findall(r"[a-z0-9]+", c.lower())})
    idx = {w: i for i, w in enumerate(vocab)}
    def embed(text):
        v = np.zeros(len(vocab), dtype=np.float32)
        for w in re.findall(r"[a-z0-9]+", text.lower()):
            if w in idx: v[idx[w]] += 1
        return v
    return embed

GOLDEN = [   # (query, keyword that marks the correct chunk)
    ("maximum loan-to-value ratio residential", "95"),
    ("minimum age for a personal loan",         "18"),
    ("debt-to-income ratio limit for approval", "45"),
    ("overdraft facility cap standard account", "2000"),
]

def evaluate(chunks, k=3):
    embed = make_embedder(chunks)
    cvecs = [embed(c) for c in chunks]
    recall = mrr = 0.0
    for q, gold in GOLDEN:
        qv = embed(q)
        ranked = sorted(range(len(chunks)), key=lambda i: cosine_similarity(qv, cvecs[i]), reverse=True)
        relevant = {i for i, c in enumerate(chunks) if gold in c}
        hit_ranks = [r + 1 for r, i in enumerate(ranked) if i in relevant]
        recall += any(r <= k for r in hit_ranks)
        mrr += (1 / hit_ranks[0]) if hit_ranks else 0
    n = len(GOLDEN)
    return recall / n, mrr / n

print(f'{"strategy":11} {"chunks":>6} {"recall@3":>9} {"MRR":>6}')
for name, fn in [("fixed", fixed_chunks), ("recursive", recursive_chunks), ("semantic", semantic_chunks)]:
    r, m = evaluate(fn(DOC))
    print(f"{name:11} {len(fn(DOC)):>6} {r:>9.2f} {m:>6.2f}")

strategy    chunks  recall@3    MRR
fixed            6      1.00   0.71
recursive        8      1.00   0.88
semantic         8      1.00   0.88


☝ All three eventually retrieve the fact (`recall@3 = 1.0`), but **MRR** separates them: fixed-size ranks the right chunk lower (the split fact dilutes the match) while recursive/semantic put it **first** more often. MRR is the metric that catches a chunking regression a pass/fail recall would miss — run it on every ingestion change.

## 4. Advanced RAG — query expansion rescues a failing query

Naive RAG passes the user's words straight to the vector search. Real queries use **abbreviations** the documents spell out — `DTI` vs *debt-to-income*. The embedding of `DTI` doesn't match the chunk, so retrieval lands on the wrong one. A **pre-retrieval** step expands known abbreviations before searching.

In [3]:
chunks = semantic_chunks(DOC)
embed  = make_embedder(chunks)
cvecs  = [embed(c) for c in chunks]

def top_chunk(query):
    qv = embed(query)
    best = max(range(len(chunks)), key=lambda i: cosine_similarity(qv, cvecs[i]))
    return best, chunks[best]

SYNONYMS = {"dti": "debt to income", "ltv": "loan to value ratio"}
def expand(query):
    extra = [v for k, v in SYNONYMS.items() if k in query.lower().split()]
    return " ".join([query, *extra])

q = "DTI cap"
i_naive, c_naive = top_chunk(q)
i_exp,   c_exp   = top_chunk(expand(q))
print(f"query          : {q!r}")
print(f"naive    -> chunk {i_naive}: {c_naive}")
print(f"expanded : {expand(q)!r}")
print(f"expanded -> chunk {i_exp}: {c_exp}")

query          : 'DTI cap'
naive    -> chunk 0: Residential mortgages require a minimum deposit of five percent.
expanded : 'DTI cap debt to income'
expanded -> chunk 4: The debt-to-income ratio must not exceed 45 percent for approval.


☝ Naive retrieval on `DTI cap` matches `cap`→ the wrong rule; expanding `DTI → debt to income` **before** searching lands the correct chunk. This pre-retrieval step (plus hybrid keyword+vector search and post-retrieval reranking) is the core of Advanced RAG — small modules bolted around the naive core, each fixing one failure mode.

## 5. Choosing an embedding model

The embedder decides retrieval quality *and* cost. The two you'll weigh at Barclays:

| | **Titan Text Embeddings v2** (Bedrock) | **text-embedding-3-small** (OpenAI) |
|---|---|---|
| Dimensions | 1024 / 512 / 256 (configurable) | 1536 (truncatable) |
| Home | **in Bedrock** — no data leaves AWS | external API |
| Cost | low, per-token | low, per-token |
| Pick when | data residency / all-AWS stack (**default here**) | already on OpenAI, need its recall |

**Rule:** match the **query** and **document** embedder (same model), and pick dimensions for the memory budget you costed on this morning's `float32` maths — 256-d Titan is 4× smaller than 1024-d for a small recall cost. Data residency usually makes **Titan** the answer inside a bank.

## 6. Modular RAG — the pipeline as swappable steps

Advanced RAG's fixes become **modules** you can reorder or swap — exactly the pipeline the Day-18 capstone graph runs. Written plainly, retrieval is a list of steps:

In [4]:
def modular_rag(query, k=2):
    steps = []
    # --- pre-retrieval module: expand ---
    q = expand(query); steps.append(("expand", q))
    # --- retrieval module: vector search ---
    qv = embed(q)
    ranked = sorted(range(len(chunks)), key=lambda i: cosine_similarity(qv, cvecs[i]), reverse=True)
    steps.append(("retrieve", ranked[:k*2]))
    # --- post-retrieval module: rerank + keep top-k ---
    top = ranked[:k]; steps.append(("rerank->topk", top))
    context = [chunks[i] for i in top]
    return context, steps

ctx, steps = modular_rag("DTI cap")
for name, out in steps:
    print(f"{name:14} -> {out}")
print("\ncontext passed to the prompt:")
for c in ctx: print(" |", c)

expand         -> DTI cap debt to income
retrieve       -> [4, 5, 2, 1]
rerank->topk   -> [4, 5]

context passed to the prompt:
 | The debt-to-income ratio must not exceed 45 percent for approval.
 | A credit score below 580 is referred to manual underwriting.


☝ Each stage is an independent module — swap `expand` for a hybrid search, add a compression step, route by query type — without touching the others. That's the Day-18 `StateGraph` in miniature: **Modular RAG is RAG as a graph of swappable nodes.**

## 7. Recap — engineering what the prompt sees

| Piece | Why it matters |
|---|---|
| **Naive → Advanced → Modular** | additive generations — each adds fixes, then makes them pluggable |
| **chunking strategy** | boundaries cap what can be retrieved; recursive/semantic keep facts whole |
| **recall@k** | did we retrieve the right chunk at all? |
| **MRR** | did we rank it first? — catches regressions recall misses |
| **query expansion** | pre-retrieval fix for abbreviations/vocabulary mismatch |
| **embedding model + dims** | retrieval quality vs memory/cost/residency (Titan default in a bank) |
| **Modular RAG** | retrieval as swappable steps — the Day-18 graph |

**Your 2.5 hrs today**
- [ ] Chunk a document three ways; eyeball where fixed-size cuts a fact.
- [ ] Score each chunker with `recall@k` and **MRR** on a golden set; explain why MRR differs.
- [ ] Add query expansion; find a query it rescues from the wrong chunk.
- [ ] Cost 256-d vs 1024-d embeddings with this morning's `float32` maths.
- [ ] Sketch your retrieval as modular steps.

**Barclays lens:** the capstone's hallucination rate is set *here* — before the model ever runs. Good chunking keeps each policy rule retrievable as a unit, MRR tells you a chunking change actually helped, query expansion closes the gap between how customers ask (`DTI`) and how policy is written (`debt-to-income`), and Titan keeps embeddings inside AWS. Retrieval quality is the ceiling on answer quality — this is where you raise it.
